# Kaggle Playground S6E8: Predicting Smartphone Addiction
## Attempt 4: Model Blending & Ensembling (LightGBM + XGBoost)

### Experiment Summary
- **Ensemble Strategy**: Weighted Probability Blending & Percentile Rank Averaging
- **Input Models**: Attempt 2 (LightGBM Classifier) + Attempt 3 (XGBoost Classifier)
- **Validation Dataset**: 5-Fold Stratified Out-Of-Fold (OOF) predictions (691,369 samples)
- **Test Dataset**: 296,302 test samples
- **Target Metric**: Out-of-Fold (OOF) ROC-AUC
- **Primary Goal**: Combine predictions from distinct GBDT algorithm families to minimize prediction variance and maximize local CV ROC-AUC for the final competition submission.

---
### 1. Environment Setup, Path Configuration & Data Ingestion
Import essential mathematical, statistical, optimization, and evaluation packages (`numpy`, `pandas`, `scipy.stats`, `scipy.optimize`, `sklearn.metrics`). 

Set directory paths for raw training data (`Data/Raw`) and processed outputs (`Data/Processed`). Load the ground-truth target array (`addicted_label`) from `train.csv`, as well as the Out-Of-Fold (OOF) arrays and test set predictions generated by Attempt 2 (LightGBM) and Attempt 3 (XGBoost).

In [2]:
import os
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, rankdata
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score

RAW_DIR = "../Data/Raw"
PROCESSED_DIR = "../Data/Processed"
TARGET_COL = "addicted_label"

train_df = pd.read_csv(os.path.join(RAW_DIR, "train.csv"), usecols=[TARGET_COL])
y_true = train_df[TARGET_COL].values

oof_lgb_path = os.path.join(PROCESSED_DIR, "oof_lgb_attempt2.npy")
oof_xgb_path = os.path.join(PROCESSED_DIR, "oof_xgb_attempt3.npy")

test_lgb_path = os.path.join(PROCESSED_DIR, "submission_attempt2_lgb.csv")
test_xgb_path = os.path.join(PROCESSED_DIR, "submission_attempt3_xgb.csv")

if os.path.exists(oof_lgb_path):
    oof_lgb = np.load(oof_lgb_path)
else:
    oof_lgb = pd.read_csv(test_lgb_path)[TARGET_COL].values

if os.path.exists(oof_xgb_path):
    oof_xgb = np.load(oof_xgb_path)
else:
    oof_xgb = pd.read_csv(test_xgb_path)[TARGET_COL].values

test_lgb_df = pd.read_csv(test_lgb_path)
test_xgb_df = pd.read_csv(test_xgb_path)

test_lgb = test_lgb_df[TARGET_COL].values
test_xgb = test_xgb_df[TARGET_COL].values
test_ids = test_lgb_df["id"].values

print("y_true shape:", y_true.shape)
print("oof_lgb shape:", oof_lgb.shape)
print("oof_xgb shape:", oof_xgb.shape)
print("test_lgb shape:", test_lgb.shape)
print("test_xgb shape:", test_xgb.shape)

y_true shape: (691369,)
oof_lgb shape: (296302,)
oof_xgb shape: (296302,)
test_lgb shape: (296302,)
test_xgb shape: (296302,)


---
### 2. Baseline Model Verification & Prediction Correlation Analysis
Evaluate individual out-of-fold (OOF) ROC-AUC validation performance for Attempt 2 (LightGBM) and Attempt 3 (XGBoost) against the ground-truth target.

Compute the Pearson correlation coefficient ($r$) between the prediction vectors:
$$\text{Pearson Correlation } r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}}$$

A lower correlation ($r < 0.98$) indicates high model diversity and significant potential for ensembling gains.

In [3]:
if len(oof_lgb) == len(y_true) and len(oof_xgb) == len(y_true):
    lgb_auc = roc_auc_score(y_true, oof_lgb)
    xgb_auc = roc_auc_score(y_true, oof_xgb)
    print("LightGBM OOF ROC-AUC:", round(lgb_auc, 6))
    print("XGBoost OOF ROC-AUC:", round(xgb_auc, 6))
    corr, _ = pearsonr(oof_lgb, oof_xgb)
    print("Pearson Correlation (OOF):", round(corr, 6))
else:
    corr, _ = pearsonr(test_lgb, test_xgb)
    print("Pearson Correlation (Test Predictions):", round(corr, 6))

Pearson Correlation (Test Predictions): 0.997994


---
### 3. Optimal Weighted Probability Blending
Optimize the blending weight $w^* \in [0.0, 1.0]$ for the linear probability combination:
$$\text{Blend OOF} = w \cdot \text{OOF}_{\text{LGB}} + (1 - w) \cdot \text{OOF}_{\text{XGB}}$$

We run a two-stage optimization process:
1. **Grid Search**: Evaluate 101 weight steps from $w = 0.00$ to $w = 1.00$ with step size 0.01.
2. **SciPy Optimizer**: Run `scipy.optimize.minimize` using the Nelder-Mead algorithm on the negative ROC-AUC objective function to obtain the precise continuous weight $w^*$.

The optimal blended score is then benchmarked against the best individual single model score.

In [4]:
best_w = 0.5
best_auc = 0.0

if len(oof_lgb) == len(y_true) and len(oof_xgb) == len(y_true):
    weights = np.linspace(0, 1, 101)
    for w in weights:
        blend_oof = w * oof_lgb + (1 - w) * oof_xgb
        score = roc_auc_score(y_true, blend_oof)
        if score > best_auc:
            best_auc = score
            best_w = w

    def objective(w):
        blend = w[0] * oof_lgb + (1 - w[0]) * oof_xgb
        return -roc_auc_score(y_true, blend)

    res = minimize(objective, [0.5], bounds=[(0, 1)], method="Nelder-Mead")
    opt_w = res.x[0]
    opt_auc = -res.fun

    print("Grid Search Best Weight (LightGBM):", round(best_w, 4))
    print("Grid Search Best OOF ROC-AUC:", round(best_auc, 6))
    print("SciPy Optimizer Best Weight (LightGBM):", round(opt_w, 4))
    print("SciPy Optimizer Best OOF ROC-AUC:", round(opt_auc, 6))
    print("Single Model Best OOF ROC-AUC:", round(max(lgb_auc, xgb_auc), 6))
    print("AUC Improvement over Best Single Model:", round(opt_auc - max(lgb_auc, xgb_auc), 6))
else:
    best_w = 0.5
    print("OOF predictions not loaded. Using default weight w = 0.5")

OOF predictions not loaded. Using default weight w = 0.5


---
### 4. Rank Averaging (Percentile Blending)
Transform probability predictions into uniform percentile ranks in the range $[0.0, 1.0]$ using `scipy.stats.rankdata`:
$$\text{Rank Blend} = w^* \cdot \text{Rank}(\text{LGB}) + (1 - w^*) \cdot \text{Rank}(\text{XGB})$$

Because ROC-AUC depends purely on the relative ordering of predictions rather than absolute probability scales, rank averaging eliminates calibration differences between LightGBM and XGBoost outputs.

The OOF ROC-AUC of Rank Averaging is computed and automatically compared against Weighted Probability Blending to determine the winning approach.

In [5]:
if len(oof_lgb) == len(y_true) and len(oof_xgb) == len(y_true):
    rank_oof_lgb = rankdata(oof_lgb) / len(oof_lgb)
    rank_oof_xgb = rankdata(oof_xgb) / len(oof_xgb)

    rank_blend_oof = opt_w * rank_oof_lgb + (1 - opt_w) * rank_oof_xgb
    rank_blend_auc = roc_auc_score(y_true, rank_blend_oof)

    print("Optimal Weight (LightGBM):", round(opt_w, 4))
    print("Weighted Probability Blend OOF ROC-AUC:", round(opt_auc, 6))
    print("Rank Averaged Blend OOF ROC-AUC:      ", round(rank_blend_auc, 6))
    print("Difference (Rank - Probability):       ", round(rank_blend_auc - opt_auc, 6))

    if rank_blend_auc > opt_auc:
        print("Selection: Rank Averaging gives higher OOF ROC-AUC.")
        use_rank = True
    else:
        print("Selection: Probability Blending gives higher OOF ROC-AUC.")
        use_rank = False
else:
    use_rank = False
    print("OOF predictions missing. Defaulting to Probability Blend.")

OOF predictions missing. Defaulting to Probability Blend.


---
### 5. Final Test Set Blending, Export & Integrity Verification
Apply the optimal blending configuration ($w^*$) to the test predictions of Attempt 2 and Attempt 3.

Format the submission DataFrame with required columns (`id`, `addicted_label`) and export the CSV file to `Data/Processed/submission_attempt4_blend.csv`.

Perform mandatory sanity verification checks:
- Verify exact test shape matching: `(296302, 2)`
- Verify `0` missing/null prediction values
- Verify probability values remain strictly within $[0.0, 1.0]$

In [6]:
sub_path = os.path.join(PROCESSED_DIR, "submission_attempt4_blend.csv")

if 'use_rank' in locals() and use_rank:
    rank_test_lgb = rankdata(test_lgb) / len(test_lgb)
    rank_test_xgb = rankdata(test_xgb) / len(test_xgb)
    final_test_preds = opt_w * rank_test_lgb + (1 - opt_w) * rank_test_xgb
else:
    w = opt_w if 'opt_w' in locals() else best_w
    final_test_preds = w * test_lgb + (1 - w) * test_xgb

sub_df = pd.DataFrame({
    "id": test_ids,
    TARGET_COL: final_test_preds
})

sub_df.to_csv(sub_path, index=False)

print("Saved submission file to:", sub_path)
print("Submission Shape:", sub_df.shape)
print("Expected Shape:   (296302, 2)")
print("\nFirst 5 Rows:")
print(sub_df.head())

print("\n--- Integrity Verification ---")
print("Null Values Count:", sub_df[TARGET_COL].isnull().sum())
print("Minimum Value:    ", round(sub_df[TARGET_COL].min(), 6))
print("Maximum Value:    ", round(sub_df[TARGET_COL].max(), 6))
print("Shape Check Passed:", sub_df.shape == (296302, 2))

Saved submission file to: ../Data/Processed\submission_attempt4_blend.csv
Submission Shape: (296302, 2)
Expected Shape:   (296302, 2)

First 5 Rows:
       id  addicted_label
0  691369        0.999555
1  691370        0.945225
2  691371        0.957513
3  691372        0.989089
4  691373        0.997706

--- Integrity Verification ---
Null Values Count: 0
Minimum Value:     0.00018
Maximum Value:     1.0
Shape Check Passed: True
